# Cell-Type Representation and Marker Validation

1. Heatmap of cell-type proportions per sample (`*` marks proportions above a threshold).
2. Marker-gene dotplot across all cells.
3. The same dotplot per treatment stage.

In [ ]:
# !pip install --quiet scanpy seaborn
# %cd <project>/notebooks

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

## Config

In [ ]:
input_path = "./../snRNAseq/output/cellbender/cleanup/scVI_integrated_adata_clean_latent20_major_cell_type_inferCNV_add_copyKAT_score_ctype_new.h5ad"

figures_output_dir = "./../figures-cellbender/integration/3_annotation/2_clean/Sup_Fig"

label_col = "cell_type_new"
sample_col = "ID_2"
stage_col = "recist_v3"
counts_layer = "raw_counts"

cell_type_order = ["Malignant", "Non-malignant epithelial", "Alveolar cell type II", "Transitional",
                   "Alveolar cell type I", "Ciliated epithelial", "Club cell", "Goblet cell",
                   "Endothelial", "Fibroblast", "Myeloid", "Lymphoid", "Cholangiocyte", "Hepatocyte"]

# marker panels (lists keep the gene order fixed in the plot)
marker_genes = {
    "Tumour": ["NAPSA", "NKX2-1", "CEACAM5", "CECR2", "MUC21"],
    "AT2": ["SFTPC", "SFTPD", "AGBL1"],
    "Transitional": ["KRT17", "TP63", "KRT8"],
    "Basal": ["KRT5"],
    "AT1": ["AGER", "CLDN18", "CLIC5", "EMP2"],
    "Ciliated": ["PIFO", "HYDIN", "LMNTD1"],
    "Club": ["SCGB1A1", "SCGB3A2"],
    "Goblet": ["MUC5AC", "MUC5B", "BPIFB1"],
    "Endothelial": ["CD34", "PECAM1", "VWF"],
    "Fibroblast": ["ACTA2", "COL6A2", "PDGFRB"],
    "Myeloid": ["MERTK", "CD163", "MARCO", "FABP4", "CSF1R"],
    "B cell": ["MS4A1"],
    "T cell": ["CD3E", "CD3D", "TCF7", "IL7R", "ITGAE", "ITGB2", "CXCL13"],
    "Cholangiocyte": ["CFTR", "SOX9"],
    "Hepatocyte": ["ALB", "AFP", "TTR"],
}


os.makedirs(figures_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80)

## 1. Load and normalise once

In [ ]:
adata = sc.read(input_path)
adata.uns.pop("log1p", None)
adata.X = adata.layers[counts_layer].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

missing = [g for v in marker_genes.values() for g in v if g not in adata.var_names]
if missing:
    print(f"marker genes not found, skipped: {missing}")
    marker_genes = {k: [g for g in v if g in adata.var_names] for k, v in marker_genes.items()}
    marker_genes = {k: v for k, v in marker_genes.items() if v}
adata.obs[label_col].value_counts()

## 2. Cell-type proportions per sample

Percent of each sample's cells (all cell types in the denominator); `*` marks values above
`star_threshold`.

In [ ]:
# --- 1. proportion heatmap ---
heatmap_cell_types = ["Malignant", "Myeloid", "Lymphoid", "Fibroblast", "Endothelial", "Alveolar cell type II",
                      "Alveolar cell type I", "Transitional", "Club cell", "Ciliated epithelial", "Goblet cell",
                      "Non-malignant epithelial"]
star_threshold = 1.0                    # % of the sample's cells; cells above are marked with *
sample_order = [
    "TN008_AN", "TN004_AN", "TN006_AN", "TN007_AN", "TN001_AN", "TN002_AN", "TN005_AN", "TN003_AN", "TN009_AN",
    "TN008", "TN004", "TN006", "TN007", "TN001", "TN002", "TN005", "TN003", "TN009",
    "P003", "P002", "P006", "P008", "P012", "P005", "P013", "P001", "P009", "P011",
    "O001", "E001", "D003", "D002", "D001",
    "R006", "R003", "R001", "R012", "R007", "R004", "R009", "R005", "R002", "R011", "R010", "R008",
]


counts = pd.crosstab(adata.obs[sample_col], adata.obs[label_col])
order = [s for s in sample_order if s in counts.index] + [s for s in counts.index if s not in sample_order]
percent = counts.div(counts.sum(axis=1), axis=0).mul(100).loc[order]
percent = percent.reindex(columns=heatmap_cell_types, fill_value=0).T

fig, ax = plt.subplots(figsize=(15, 5))
sns.heatmap(percent, cmap="YlOrRd", vmin=0, vmax=100, linewidths=0.5, ax=ax,
            cbar_kws={"label": "% of cells per sample"})
for y, x in zip(*(percent.values > star_threshold).nonzero()):
    ax.text(x + 0.5, y + 0.5, "*", ha="center", va="center", fontsize=10)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/cell_type_proportion_by_sample_{star_threshold:g}percent.pdf", bbox_inches="tight")
plt.show()
percent.round(2).to_csv(f"{figures_output_dir}/cell_type_proportion_by_sample.csv")

## 3. Marker dotplots (all cells and per treatment stage)

Gene names are italicised. Per-stage plots exclude liver cells (`dotplot_exclude_per_stage`).

In [ ]:
# --- 2-3. dotplots: output file -> stages included (None = all cells) ---
dotplots = {
    "dotplot_markers_all_sample.pdf": None,
    "dotplot_markers_adj_normal.pdf": ["0_adj_normal"],
    "dotplot_markers_treatment_naive.pdf": ["1_treatment_naive"],
    # "dotplot_markers_on_treatment.pdf": ["3_on_treatment"],
    # "dotplot_markers_TKI_resistance.pdf": ["4_TKI_resistance"],
}
dotplot_exclude_per_stage = ["Hepatocyte", "Cholangiocyte"]   # liver cells only in resistant biopsies

for fname, stages in dotplots.items():
    if stages is None:
        sub = adata
    else:
        sub = adata[adata.obs[stage_col].isin(stages) & ~adata.obs[label_col].isin(dotplot_exclude_per_stage)]
    order = [c for c in cell_type_order if c in set(sub.obs[label_col])]
    print(f"{fname}: {sub.n_obs} cells")
    axes = sc.pl.dotplot(sub, marker_genes, groupby=label_col, categories_order=order, standard_scale="var",
                         cmap="Blues", swap_axes=True, figsize=(5, 9 if stages else 12), show=False)
    for label in axes["mainplot_ax"].get_yticklabels():
        label.set_style("italic")
    plt.savefig(f"{figures_output_dir}/{fname}", bbox_inches="tight")
    plt.show()